# 02 — TSI Analysis & Experiments
## Sensibilidad Temporal de Descriptores de Audio Artesanales

Este notebook carga los features pre-extraídos (del notebook `01_feature_extraction.ipynb`) y ejecuta:
1. Cómputo del Índice de Sensibilidad Temporal (TSI)
2. Comparación de 4 estrategias de fusión
3. Experimento multi-label (MTAT)
4. Análisis de importancia (Permutation Importance)
5. Validación estadística (Wilcoxon, Cliff's delta, Spearman)
6. Visualizaciones y tablas LaTeX para el paper

**Requisito:** Ejecutar primero `01_feature_extraction.ipynb` para generar los `.npy` en Drive.

**Ejecución:** Google Colab Pro (GPU recomendada para MLP)

## 0. Setup & Load Cached Features

In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Install dependencies
!pip install -q librosa scikit-learn torch torchaudio tqdm seaborn

# Clone the repo to get the src/ modules
import os
REPO_URL = "https://github.com/Gabrieleeh32159/my_paper.git"
REPO_DIR = "/content/my_paper"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

Cloning into '/content/my_paper'...
remote: Enumerating objects: 227, done.
remote: Counting objects: 100% (227/227), done.
remote: Compressing objects: 100% (155/155), done.
remote: Total 227 (delta 142), reused 145 (delta 72), pack-reused 0 (from 0)
Receiving objects: 100% (227/227), 21.83 MiB | 21.30 MiB/s, done.
Resolving deltas: 100% (142/142), done.


In [3]:
import os
import sys
import numpy as np
import json
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# === CONFIGURATION ===
REPO_DIR = Path('/content/my_paper')

# Google Drive paths
DRIVE_ROOT = Path('/content/drive/MyDrive/tsi_experiments')
DATA_ROOT = DRIVE_ROOT / 'data'
FEATURES_ROOT = DRIVE_ROOT / 'features'
RESULTS_ROOT = DRIVE_ROOT / 'results'

RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

# Add repo's experiments/ to path
sys.path.insert(0, str(REPO_DIR / 'experiments'))

# Dataset paths (needed for metadata: n_classes, class_names, task_type)
DATASET_PATHS = {
    'gtzan': DATA_ROOT / 'gtzan',
    'fma_small': DATA_ROOT,
    'mtat': DATA_ROOT / 'magnatagatune',
    'irmas': DATA_ROOT / 'irmas',
}

SEED = 42
np.random.seed(SEED)

print(f"Features source: {FEATURES_ROOT}")
print(f"Results output:  {RESULTS_ROOT}")

Features source: /content/drive/MyDrive/tsi_experiments/features
Results output:  /content/drive/MyDrive/tsi_experiments/results


In [4]:
%cd {REPO_DIR}/experiments

from src.features import FEATURE_DIMS, SCALES, TRACK_DIM
from src.data_loader import get_dataset
from src.fusion import early_fusion
from src.classifiers import get_classifier, RFClassifier
from src.tsi import compute_tsi, tsi_consistency
from src.evaluation import run_full_evaluation
from src.importance import aggregate_importance_by_descriptor_and_scale
from src.stats import spearman_with_bootstrap_ci, run_pairwise_comparisons

print("All modules loaded.")
print(f"Feature dimensions per frame: {sum(FEATURE_DIMS.values())} ({FEATURE_DIMS})")
print(f"Track vector dim per scale: {TRACK_DIM}")
print(f"Temporal scales: {SCALES}")

/content/my_paper/experiments
All modules loaded.
Feature dimensions per frame: 48 ({'mfcc': 20, 'chroma': 12, 'spectral_centroid': 1, 'spectral_contrast': 7, 'spectral_rolloff': 1, 'zcr': 1, 'tonnetz': 6})
Track vector dim per scale: 192
Temporal scales: {'short': 0.2, 'medium': 2.0, 'long': 5.0}


In [5]:
# Load dataset metadata (for n_classes, class_names, task_type)
datasets = {}

for name, path in DATASET_PATHS.items():
    try:
        datasets[name] = get_dataset(name, str(path))
        print(f"{name}: {len(datasets[name])} items, {datasets[name].n_classes} classes, task={datasets[name].task_type}")
    except Exception as e:
        print(f"{name}: not available ({e})")

print(f"\nLoaded metadata for: {list(datasets.keys())}")

gtzan: 1000 items, 10 classes, task=multiclass
fma_small: 8000 items, 8 classes, task=multiclass
mtat: 25863 items, 50 classes, task=multilabel
irmas: 7512 items, 11 classes, task=multiclass

Loaded metadata for: ['gtzan', 'fma_small', 'mtat', 'irmas']


In [ ]:
# Load pre-extracted features from Drive
all_features = {}
all_labels = {}
all_splits = {}

for name in datasets.keys():
    short_path = FEATURES_ROOT / f"{name}_short.npy"
    if not short_path.exists():
        print(f"{name}: features not found in {FEATURES_ROOT}, skipping.")
        print(f"  Run 01_feature_extraction.ipynb first.")
        continue

    all_features[name] = {
        'short':  np.load(FEATURES_ROOT / f"{name}_short.npy"),
        'medium': np.load(FEATURES_ROOT / f"{name}_medium.npy"),
        'long':   np.load(FEATURES_ROOT / f"{name}_long.npy"),
    }
    raw_labels = np.load(FEATURES_ROOT / f"{name}_labels.npy", allow_pickle=True)
    all_splits[name] = np.load(FEATURES_ROOT / f"{name}_splits.npy", allow_pickle=True)

    # Fix: labels were saved as dtype=object, which sklearn rejects.
    # Cast to proper numeric types.
    if datasets[name].task_type == 'multilabel':
        all_labels[name] = np.stack(raw_labels).astype(np.float32)
    else:
        all_labels[name] = raw_labels.astype(int)

    n = all_features[name]['short'].shape[0]
    d = all_features[name]['short'].shape[1]
    print(f"{name}: {n} tracks x {d}-d x 3 scales (labels dtype={all_labels[name].dtype})")

print(f"\n=== Features loaded for: {list(all_features.keys())} ===")

gtzan: 999 tracks x 192-d x 3 scales loaded
fma_small: 7996 tracks x 192-d x 3 scales loaded
mtat: 25700 tracks x 192-d x 3 scales loaded
irmas: 3756 tracks x 192-d x 3 scales loaded

=== Features loaded for: ['gtzan', 'fma_small', 'mtat', 'irmas'] ===


## 1. TSI Computation

For each descriptor, train a classifier using ONLY that descriptor at each scale.

$$TSI(f) = \frac{\max_k Acc(f,k) - \min_k Acc(f,k)}{Acc_{chance}}$$

In [7]:
def get_train_test_indices(splits, labels, dataset_name):
    """Get train/test indices based on dataset splits."""
    if dataset_name == 'fma_small':
        train_idx = np.where((splits == 'train') | (splits == 'val'))[0]
        test_idx = np.where(splits == 'test')[0]
    elif dataset_name == 'mtat':
        train_idx = np.where(splits == 'train')[0]
        test_idx = np.where(splits == 'test')[0]
    elif dataset_name == 'irmas':
        train_idx = np.where(splits == 'train')[0]
        test_idx = np.where(splits == 'test')[0]
    else:  # gtzan - use first fold for initial TSI
        from sklearn.model_selection import StratifiedKFold
        skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=SEED)
        folds = list(skf.split(np.zeros(len(labels)), labels))
        train_idx, test_idx = folds[0]
    return train_idx, test_idx

In [8]:
# Compute TSI for all available datasets with Random Forest
tsi_results = {}

for ds_name in all_features.keys():
    if ds_name == 'mtat':
        # Skip multilabel for TSI (accuracy not well-defined)
        continue

    print(f"\n{'='*60}")
    print(f"Computing TSI for {ds_name} (n_classes={datasets[ds_name].n_classes})")
    print(f"{'='*60}")

    train_idx, test_idx = get_train_test_indices(
        all_splits[ds_name], all_labels[ds_name], ds_name
    )
    print(f"  Train: {len(train_idx)}, Test: {len(test_idx)}")

    tsi_scores, acc_matrix, opt_scales = compute_tsi(
        features=all_features[ds_name],
        labels=all_labels[ds_name],
        n_classes=datasets[ds_name].n_classes,
        classifier_name='rf',
        train_idx=train_idx,
        test_idx=test_idx,
    )

    tsi_results[ds_name] = {
        'tsi_scores': tsi_scores,
        'accuracy_matrix': acc_matrix,
        'optimal_scales': opt_scales,
    }

    # Print results
    print(f"\n  {'Descriptor':<20} {'TSI':>6} {'Short':>8} {'Medium':>8} {'Long':>8} {'Optimal':>8}")
    print(f"  {'-'*60}")
    for desc in FEATURE_DIMS.keys():
        tsi = tsi_scores[desc]
        acc_s = acc_matrix[desc]['short']
        acc_m = acc_matrix[desc]['medium']
        acc_l = acc_matrix[desc]['long']
        opt = opt_scales[desc]
        print(f"  {desc:<20} {tsi:>6.3f} {acc_s:>8.4f} {acc_m:>8.4f} {acc_l:>8.4f} {opt:>8}")

# Save TSI results
tsi_save = {}
for ds_name, res in tsi_results.items():
    tsi_save[ds_name] = {
        'tsi_scores': res['tsi_scores'],
        'accuracy_matrix': res['accuracy_matrix'],
        'optimal_scales': res['optimal_scales'],
    }

with open(RESULTS_ROOT / 'tsi_results.json', 'w') as f:
    json.dump(tsi_save, f, indent=2)
print(f"\nTSI results saved to {RESULTS_ROOT / 'tsi_results.json'}")


Computing TSI for gtzan (n_classes=10)


ValueError: Supported target types are: ('binary', 'multiclass'). Got 'unknown' instead.

## 2. Fusion Strategy Comparison

Compare 6 strategies: single-scale (short/medium/long), early fusion, late fusion, TSI-weighted fusion.

In [ ]:
def run_experiment_multiclass(ds_name, features, labels, splits, dataset, tsi_info, classifier_name='rf'):
    """
    Run full experiment for a multiclass dataset.
    Tests all 6 strategies with the given classifier.
    """
    train_idx, test_idx = get_train_test_indices(splits, labels, ds_name)
    n_classes = dataset.n_classes
    strategies = ['short', 'medium', 'long', 'early', 'late', 'tsi_weighted']
    results = {}

    for strategy in strategies:
        print(f"    Strategy: {strategy}...", end=' ')

        if strategy in ('short', 'medium', 'long'):
            input_dim = TRACK_DIM  # 192
        elif strategy == 'early':
            input_dim = TRACK_DIM * 3  # 576
        elif strategy == 'tsi_weighted':
            input_dim = sum(4 * FEATURE_DIMS[d] for d in FEATURE_DIMS.keys())  # 192
        else:  # late
            input_dim = TRACK_DIM  # 192

        clf = get_classifier(classifier_name, input_dim=input_dim, n_classes=n_classes)

        fusion_params = None
        if strategy == 'tsi_weighted' and tsi_info:
            fusion_params = {
                'tsi_scores': tsi_info['tsi_scores'],
                'optimal_scales': tsi_info['optimal_scales'],
            }

        result = run_full_evaluation(
            features=features,
            labels=labels,
            classifier=clf,
            train_idx=train_idx,
            test_idx=test_idx,
            strategy=strategy,
            task_type='multiclass',
            class_names=dataset.class_names,
            fusion_params=fusion_params,
        )
        results[strategy] = result
        print(f"Acc={result['accuracy']:.4f}, F1={result['f1_macro']:.4f}")

    return results

In [ ]:
# Run experiments for all datasets x classifiers
all_experiment_results = {}
CLASSIFIERS = ['rf', 'svm', 'mlp']

for ds_name in all_features.keys():
    if datasets[ds_name].task_type == 'multilabel':
        continue  # Handle MTAT separately

    print(f"\n{'='*70}")
    print(f"DATASET: {ds_name} ({datasets[ds_name].n_classes} classes, {all_features[ds_name]['short'].shape[0]} tracks)")
    print(f"{'='*70}")

    all_experiment_results[ds_name] = {}
    tsi_info = tsi_results.get(ds_name)

    for clf_name in CLASSIFIERS:
        print(f"\n  Classifier: {clf_name.upper()}")
        print(f"  {'-'*50}")

        results = run_experiment_multiclass(
            ds_name=ds_name,
            features=all_features[ds_name],
            labels=all_labels[ds_name],
            splits=all_splits[ds_name],
            dataset=datasets[ds_name],
            tsi_info=tsi_info,
            classifier_name=clf_name,
        )
        all_experiment_results[ds_name][clf_name] = results

# Save results
results_save = {}
for ds_name, clf_results in all_experiment_results.items():
    results_save[ds_name] = {}
    for clf_name, strat_results in clf_results.items():
        results_save[ds_name][clf_name] = {}
        for strat, metrics in strat_results.items():
            results_save[ds_name][clf_name][strat] = {
                'accuracy': metrics['accuracy'],
                'f1_macro': metrics['f1_macro'],
            }

with open(RESULTS_ROOT / 'experiment_results.json', 'w') as f:
    json.dump(results_save, f, indent=2)
print(f"\nResults saved to {RESULTS_ROOT / 'experiment_results.json'}")

## 2b. MTAT (Multi-label) Experiment

In [ ]:
# MTAT experiment (if available)
if 'mtat' in all_features:
    print("\n" + "="*70)
    print("DATASET: MTAT (50 tags, multilabel)")
    print("="*70)

    features_mtat = all_features['mtat']
    labels_mtat = all_labels['mtat']
    splits_mtat = all_splits['mtat']

    train_idx = np.where(splits_mtat == 'train')[0]
    test_idx = np.where(splits_mtat == 'test')[0]

    # Compute class weights for MLP (inverse frequency)
    pos_counts = labels_mtat[train_idx].sum(axis=0)
    neg_counts = len(train_idx) - pos_counts
    class_weights = neg_counts / (pos_counts + 1e-6)

    mtat_results = {}
    strategies = ['short', 'medium', 'long', 'early', 'late']

    for strategy in strategies:
        print(f"  Strategy: {strategy}...", end=' ')

        if strategy in ('short', 'medium', 'long'):
            input_dim = TRACK_DIM  # 192
        else:
            input_dim = TRACK_DIM * 3  # 576

        clf = get_classifier(
            'mlp', input_dim=input_dim, n_classes=50,
            task_type='multilabel', class_weights=class_weights
        )

        result = run_full_evaluation(
            features=features_mtat,
            labels=labels_mtat,
            classifier=clf,
            train_idx=train_idx,
            test_idx=test_idx,
            strategy=strategy,
            task_type='multilabel',
            class_names=datasets['mtat'].class_names,
        )
        mtat_results[strategy] = result
        print(f"mAP={result['mAP']:.4f}, AUC={result.get('roc_auc_macro', 'N/A')}")

    # Save MTAT results
    mtat_save = {s: {'mAP': r['mAP'], 'roc_auc_macro': r.get('roc_auc_macro')}
                 for s, r in mtat_results.items()}
    with open(RESULTS_ROOT / 'mtat_results.json', 'w') as f:
        json.dump(mtat_save, f, indent=2)
else:
    print("MTAT not available, skipping multilabel experiment.")

## 3. Feature Importance Analysis

In [ ]:
# Permutation Importance on early fusion with RF
from sklearn.inspection import permutation_importance as sklearn_pi

importance_results = {}

for ds_name in all_features.keys():
    if datasets[ds_name].task_type == 'multilabel':
        continue

    print(f"\nComputing Permutation Importance for {ds_name}...")

    features = all_features[ds_name]
    labels = all_labels[ds_name]
    splits = all_splits[ds_name]

    train_idx, test_idx = get_train_test_indices(splits, labels, ds_name)

    # Train RF on early fusion
    X_early = early_fusion(features)
    X_train, X_test = X_early[train_idx], X_early[test_idx]
    y_train, y_test = labels[train_idx], labels[test_idx]

    rf = RFClassifier()
    rf.fit(X_train, y_train)

    # Permutation importance (using underlying sklearn model + scaler)
    X_test_scaled = rf.scaler.transform(X_test)
    pi_result = sklearn_pi(
        rf.model, X_test_scaled, y_test,
        n_repeats=30, scoring='f1_macro',
        random_state=SEED, n_jobs=-1
    )

    # Aggregate by descriptor x scale
    imp_matrix = aggregate_importance_by_descriptor_and_scale(
        pi_result.importances_mean, vector_dim=TRACK_DIM
    )

    importance_results[ds_name] = {
        'pi_mean': pi_result.importances_mean.tolist(),
        'pi_std': pi_result.importances_std.tolist(),
        'matrix': imp_matrix,
        'mdi': rf.feature_importances_.tolist(),
    }

    # Print importance matrix
    print(f"\n  Importance Matrix (PI, descriptor x scale):")
    print(f"  {'Descriptor':<20} {'Short':>10} {'Medium':>10} {'Long':>10}")
    print(f"  {'-'*52}")
    for desc in FEATURE_DIMS.keys():
        s = imp_matrix[desc]['short']
        m = imp_matrix[desc]['medium']
        l = imp_matrix[desc]['long']
        print(f"  {desc:<20} {s:>10.5f} {m:>10.5f} {l:>10.5f}")

# Save importance results
with open(RESULTS_ROOT / 'importance_results.json', 'w') as f:
    json.dump(importance_results, f, indent=2)
print(f"\nImportance results saved to {RESULTS_ROOT / 'importance_results.json'}")

## 4. Statistical Validation

In [ ]:
# Statistical tests: Wilcoxon + Cliff's delta
statistical_results = {}

for ds_name in all_experiment_results.keys():
    print(f"\n{'='*60}")
    print(f"Statistical Tests: {ds_name}")
    print(f"{'='*60}")

    # Use per-class F1 scores as paired observations
    clf_name = 'rf'  # Primary classifier
    strat_results = all_experiment_results[ds_name][clf_name]

    paired_scores = {}
    for strategy, metrics in strat_results.items():
        if 'f1_per_class' in metrics:
            paired_scores[strategy] = np.array(metrics['f1_per_class'])

    if len(paired_scores) < 2:
        print("  Not enough strategies with per-class F1 for pairwise tests.")
        continue

    comparisons = run_pairwise_comparisons(paired_scores)
    statistical_results[ds_name] = comparisons

    print(f"\n  {'A vs B':<30} {'p-val':>8} {'Sig?':>6} {'Cliff d':>8} {'Effect':>10}")
    print(f"  {'-'*65}")
    for comp in comparisons:
        pair = f"{comp['strategy_a']} vs {comp['strategy_b']}"
        sig = 'Y' if comp['significant'] else 'N'
        print(f"  {pair:<30} {comp['p_value']:>8.5f} {sig:>6} {comp['cliffs_delta']:>8.3f} {comp['effect_magnitude']:>10}")

# Save statistical results
with open(RESULTS_ROOT / 'statistical_tests.json', 'w') as f:
    json.dump(statistical_results, f, indent=2, default=str)
print(f"\nStatistical results saved to {RESULTS_ROOT / 'statistical_tests.json'}")

In [ ]:
# TSI Consistency: Spearman correlation between datasets
print("\n=== TSI Consistency Across Datasets ===")

if len(tsi_results) >= 2:
    tsi_for_consistency = {
        ds_name: res['tsi_scores']
        for ds_name, res in tsi_results.items()
    }

    correlations = tsi_consistency(tsi_for_consistency)

    descriptors = list(FEATURE_DIMS.keys())

    for (cfg_a, cfg_b), rho in correlations.items():
        x = np.array([tsi_for_consistency[cfg_a][d] for d in descriptors])
        y = np.array([tsi_for_consistency[cfg_b][d] for d in descriptors])
        ci_result = spearman_with_bootstrap_ci(x, y)

        consistent = 'Y' if ci_result['consistent'] else 'N'
        print(f"  {cfg_a} vs {cfg_b}: rho={ci_result['rho']:.3f} "
              f"[{ci_result['ci_lower']:.3f}, {ci_result['ci_upper']:.3f}] "
              f"Consistent(>0.7): {consistent}")
else:
    print("  Need at least 2 datasets for consistency analysis.")

## 5. Visualization & Results for Paper

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set IEEE paper style
plt.rcParams.update({
    'font.size': 10,
    'axes.labelsize': 10,
    'axes.titlesize': 11,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 9,
    'figure.figsize': (7, 4),
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
})

In [ ]:
# Figure 1: TSI Bar Chart per descriptor
n_datasets = len(tsi_results)
fig, axes = plt.subplots(1, max(n_datasets, 1), figsize=(4*max(n_datasets, 1), 4), sharey=True)
if n_datasets == 1:
    axes = [axes]

for ax, (ds_name, res) in zip(axes, tsi_results.items()):
    descriptors = list(res['tsi_scores'].keys())
    tsi_vals = [res['tsi_scores'][d] for d in descriptors]

    colors = ['#e74c3c' if v > 1.0 else '#3498db' if v > 0.5 else '#95a5a6' for v in tsi_vals]

    ax.barh(descriptors, tsi_vals, color=colors)
    ax.set_xlabel('TSI')
    ax.set_title(ds_name.upper())
    ax.axvline(x=1.0, color='red', linestyle='--', alpha=0.5, label='High sensitivity')
    ax.axvline(x=0.5, color='blue', linestyle='--', alpha=0.5, label='Moderate')

axes[0].set_ylabel('Descriptor')
plt.suptitle('Temporal Sensitivity Index (TSI) per Descriptor', fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS_ROOT / 'fig_tsi_barchart.pdf')
plt.savefig(RESULTS_ROOT / 'fig_tsi_barchart.png')
plt.show()
print(f"Saved to {RESULTS_ROOT / 'fig_tsi_barchart.pdf'}")

In [ ]:
# Figure 2: 7x3 Sensitivity Matrix (Heatmap)
for ds_name, res in tsi_results.items():
    acc_matrix = res['accuracy_matrix']
    descriptors = list(FEATURE_DIMS.keys())
    scales = list(SCALES.keys())

    matrix = np.zeros((len(descriptors), len(scales)))
    for i, desc in enumerate(descriptors):
        for j, scale in enumerate(scales):
            matrix[i, j] = acc_matrix[desc][scale]

    fig, ax = plt.subplots(figsize=(5, 5))
    sns.heatmap(
        matrix, annot=True, fmt='.3f',
        xticklabels=['Short\n(200ms)', 'Medium\n(2s)', 'Long\n(5s)'],
        yticklabels=descriptors,
        cmap='YlOrRd', ax=ax,
        vmin=matrix.min() * 0.9,
        vmax=matrix.max() * 1.05,
    )
    ax.set_title(f'Accuracy Matrix (descriptor x scale) -- {ds_name.upper()}')
    ax.set_xlabel('Temporal Scale')
    ax.set_ylabel('Descriptor')
    plt.tight_layout()
    plt.savefig(RESULTS_ROOT / f'fig_sensitivity_matrix_{ds_name}.pdf')
    plt.savefig(RESULTS_ROOT / f'fig_sensitivity_matrix_{ds_name}.png')
    plt.show()
    print(f"Saved: fig_sensitivity_matrix_{ds_name}.pdf")

In [ ]:
# Figure 3: Strategy Comparison Bar Chart
for ds_name, clf_results in all_experiment_results.items():
    strategies = ['short', 'medium', 'long', 'early', 'late', 'tsi_weighted']
    classifiers = list(clf_results.keys())

    fig, ax = plt.subplots(figsize=(10, 5))
    x = np.arange(len(strategies))
    width = 0.25

    for i, clf_name in enumerate(classifiers):
        f1_scores = [clf_results[clf_name][s]['f1_macro'] for s in strategies]
        ax.bar(x + i*width, f1_scores, width, label=clf_name.upper())

    ax.set_xlabel('Fusion Strategy')
    ax.set_ylabel('F1 Macro')
    ax.set_title(f'Strategy Comparison -- {ds_name.upper()}')
    ax.set_xticks(x + width)
    ax.set_xticklabels(strategies, rotation=20)
    ax.legend()
    ax.set_ylim(0, 1)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(RESULTS_ROOT / f'fig_strategy_comparison_{ds_name}.pdf')
    plt.savefig(RESULTS_ROOT / f'fig_strategy_comparison_{ds_name}.png')
    plt.show()

In [ ]:
# Figure 4: Importance Matrix Heatmap (early fusion, RF)
for ds_name, imp_res in importance_results.items():
    matrix = imp_res['matrix']
    descriptors = list(FEATURE_DIMS.keys())
    scales = list(SCALES.keys())

    mat_arr = np.zeros((len(descriptors), len(scales)))
    for i, desc in enumerate(descriptors):
        for j, scale in enumerate(scales):
            mat_arr[i, j] = matrix[desc][scale]

    fig, ax = plt.subplots(figsize=(5, 5))
    sns.heatmap(
        mat_arr, annot=True, fmt='.4f',
        xticklabels=['Short\n(200ms)', 'Medium\n(2s)', 'Long\n(5s)'],
        yticklabels=descriptors,
        cmap='viridis', ax=ax,
    )
    ax.set_title(f'Permutation Importance (descriptor x scale) -- {ds_name.upper()}')
    ax.set_xlabel('Temporal Scale')
    ax.set_ylabel('Descriptor')
    plt.tight_layout()
    plt.savefig(RESULTS_ROOT / f'fig_importance_matrix_{ds_name}.pdf')
    plt.savefig(RESULTS_ROOT / f'fig_importance_matrix_{ds_name}.png')
    plt.show()

## 6. LaTeX Tables for Paper

In [ ]:
# Generate LaTeX table: Main results
def generate_results_latex(results, dataset_names):
    """Generate LaTeX table for paper."""
    strategies = ['short', 'medium', 'long', 'early', 'late', 'tsi_weighted']
    strat_labels = ['Short (200ms)', 'Medium (2s)', 'Long (5s)',
                    'Early Fusion', 'Late Fusion', 'TSI-Weighted']

    lines = []
    lines.append(r'\begin{table}[htp]')
    lines.append(r'\centering')
    lines.append(r'\caption{Comparison of fusion strategies (F1 Macro, RF classifier).}')
    lines.append(r'\label{tab:results_main}')

    cols = 'l' + 'c' * len(dataset_names)
    lines.append(r'\begin{tabular}{@{}' + cols + r'@{}}')
    lines.append(r'\toprule')

    header = r'\textbf{Strategy}'
    for ds in dataset_names:
        header += f' & \\textbf{{{ds.upper()}}}'
    header += r' \\'
    lines.append(header)
    lines.append(r'\midrule')

    for strat, label in zip(strategies, strat_labels):
        row = label
        for ds in dataset_names:
            if ds in results and 'rf' in results[ds] and strat in results[ds]['rf']:
                f1 = results[ds]['rf'][strat]['f1_macro']
                row += f' & {f1:.4f}'
            else:
                row += ' & ---'
        row += r' \\'
        lines.append(row)

    lines.append(r'\bottomrule')
    lines.append(r'\end{tabular}')
    lines.append(r'\end{table}')

    return '\n'.join(lines)


latex_table = generate_results_latex(all_experiment_results, list(all_experiment_results.keys()))
print(latex_table)

with open(RESULTS_ROOT / 'table_results_main.tex', 'w') as f:
    f.write(latex_table)
print(f"\nSaved to {RESULTS_ROOT / 'table_results_main.tex'}")

In [ ]:
# Generate LaTeX table: TSI values
def generate_tsi_latex(tsi_results):
    """Generate LaTeX table for TSI values."""
    descriptors = list(FEATURE_DIMS.keys())
    datasets_available = list(tsi_results.keys())

    lines = []
    lines.append(r'\begin{table}[htp]')
    lines.append(r'\centering')
    lines.append(r'\caption{Temporal Sensitivity Index (TSI) per descriptor and dataset.}')
    lines.append(r'\label{tab:tsi_values}')

    cols = 'l' + 'c' * len(datasets_available)
    lines.append(r'\begin{tabular}{@{}' + cols + r'@{}}')
    lines.append(r'\toprule')

    header = r'\textbf{Descriptor}'
    for ds in datasets_available:
        header += f' & \\textbf{{{ds.upper()}}}'
    header += r' \\'
    lines.append(header)
    lines.append(r'\midrule')

    for desc in descriptors:
        row = desc.replace('_', ' ').title()
        for ds in datasets_available:
            tsi = tsi_results[ds]['tsi_scores'][desc]
            row += f' & {tsi:.3f}'
        row += r' \\'
        lines.append(row)

    lines.append(r'\bottomrule')
    lines.append(r'\end{tabular}')
    lines.append(r'\end{table}')

    return '\n'.join(lines)


latex_tsi = generate_tsi_latex(tsi_results)
print(latex_tsi)

with open(RESULTS_ROOT / 'table_tsi_values.tex', 'w') as f:
    f.write(latex_tsi)
print(f"\nSaved to {RESULTS_ROOT / 'table_tsi_values.tex'}")

## 7. Summary & Final Report

In [ ]:
# Print comprehensive summary
print("\n" + "="*80)
print("EXPERIMENT SUMMARY")
print("="*80)

print("\n--- TSI Rankings (Higher = More temporally sensitive) ---")
for ds_name, res in tsi_results.items():
    sorted_tsi = sorted(res['tsi_scores'].items(), key=lambda x: -x[1])
    print(f"\n  {ds_name.upper()}:")
    for i, (desc, tsi) in enumerate(sorted_tsi, 1):
        opt = res['optimal_scales'][desc]
        print(f"    {i}. {desc:<20} TSI={tsi:.3f}  (optimal: {opt})")

print("\n\n--- Best Strategy per Dataset (F1 Macro, RF) ---")
for ds_name, clf_results in all_experiment_results.items():
    rf_results = clf_results.get('rf', {})
    if rf_results:
        best = max(rf_results.items(), key=lambda x: x[1]['f1_macro'])
        print(f"  {ds_name.upper()}: {best[0]} (F1={best[1]['f1_macro']:.4f})")

print("\n\n--- Key Findings ---")
print("  1. Descriptors with HIGH temporal sensitivity (TSI > 0.5):")
for ds_name, res in tsi_results.items():
    high_tsi = [(d, v) for d, v in res['tsi_scores'].items() if v > 0.5]
    if high_tsi:
        print(f"     {ds_name}: {', '.join(d for d, _ in high_tsi)}")

print("  2. Descriptors with LOW temporal sensitivity (TSI < 0.3):")
for ds_name, res in tsi_results.items():
    low_tsi = [(d, v) for d, v in res['tsi_scores'].items() if v < 0.3]
    if low_tsi:
        print(f"     {ds_name}: {', '.join(d for d, _ in low_tsi)}")

print("\n\n--- Files Saved to Drive ---")
for f in sorted(RESULTS_ROOT.glob('*')):
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name} ({size_kb:.1f} KB)")

print("\n" + "="*80)
print("DONE. All results saved to:", RESULTS_ROOT)
print("="*80)